In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

***Fetching the Datasets***

In [2]:
train_dataset = pd.read_csv("data/train.csv")
test_dataset = pd.read_csv("data/test.csv")

# Drop the target and non-predictive columns for X
X_train = train_dataset.drop(['Survived', 'PassengerId', 'Name', 'Ticket'], axis=1)

# Only keep the target for y
y_train = train_dataset['Survived']

# Drop the target and non-predictive columns for X
X_test = test_dataset.drop(['PassengerId', 'Name', 'Ticket'], axis=1)

# Preprocessing the Datasets

In [3]:
print(f"\n{X_train.info()}")
print(f"\n{X_test.info()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    891 non-null    int64  
 1   Sex       891 non-null    object 
 2   Age       714 non-null    float64
 3   SibSp     891 non-null    int64  
 4   Parch     891 non-null    int64  
 5   Fare      891 non-null    float64
 6   Cabin     204 non-null    object 
 7   Embarked  889 non-null    object 
dtypes: float64(2), int64(3), object(3)
memory usage: 55.8+ KB

None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    418 non-null    int64  
 1   Sex       418 non-null    object 
 2   Age       332 non-null    float64
 3   SibSp     418 non-null    int64  
 4   Parch     418 non-null    int64  
 5   Fare      417 non-null    float64
 6   Cabin     91 non

In [ ]:
from sklearn.linear_model import LogisticRegression
from preprocessing_utils_pipeline_universal import get_preprocessing_pipeline

def add_titanic_features(df):
    df_copy = df.copy()
    if 'SibSp' in df_copy.columns and 'Parch' in df_copy.columns:
        df_copy['FamilySize'] = df_copy['SibSp'] + df_copy['Parch'] + 1
    # Add other Titanic-specific logic here...
    return df_copy

# Use it like this:
X_train_engineered = add_titanic_features(X_train)
pipeline = get_universal_pipeline(X_train_engineered, LogisticRegression())
pipeline.fit(X_train_engineered, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['Sex', 'Cabin', 'Embarked'], dtype='object'))])),
                ('model', LogisticRegression())])

In [ ]:
# Evaluating Logistic Regression model

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix